# Bayesian Optimisation — Function 8 (8D)

**Author:** Dibyajyoti Pradhan  
**Programme:** Imperial College London Professional Certificate in ML/AI  
**Module:** BBO Capstone (Modules 12–22)  

**Strategy note:** High output. X1, X3 dominant. Gradient ascent in pseudo-2D.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src/ to path for reusable utilities
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from bbo_utils import (
    lhs_candidates, fit_gp, gp_posterior,
    ucb_acquisition, suggest_next_query,
    dimension_sensitivity, plot_running_best,
)

np.random.seed(42)

## 2. Load Initial Data

Initial observations provided at the start of the BBO challenge.

In [ ]:
data_dir = os.path.join('..', 'data', 'initial_data', 'function_8')

X_init = np.load(os.path.join(data_dir, 'initial_inputs.npy'))
y_init = np.load(os.path.join(data_dir, 'initial_outputs.npy'))

print(f'Initial inputs shape:  {X_init.shape}')
print(f'Initial outputs shape: {y_init.shape}')
print(f'Output range: [{y_init.min():.6f}, {y_init.max():.6f}]')
print(f'Best initial output:   {y_init.max():.6f}')
print(f'Best initial input:    {X_init[np.argmax(y_init)]}')

## 3. Add Subsequent Query Data

Queries from Weeks 1–5 with confirmed outputs.  
Week 6 submitted — awaiting results (not included in training).

In [ ]:
# Weekly queries — Weeks 1–5 (confirmed outputs)
queries_X = np.array([
    [0.001000, 0.001000, 0.001000, 0.001000, 0.999000, 0.001000, 0.001000, 0.001000],   # W1
    [0.117631, 0.033906, 0.164674, 0.039646, 0.373290, 0.749774, 0.493738, 0.936259],   # W2
    [0.106675, 0.011439, 0.210614, 0.091867, 0.366575, 0.736816, 0.442316, 0.860648],   # W3
    [0.030197, 0.007025, 0.128571, 0.023762, 0.500222, 0.763973, 0.526864, 0.888811],   # W4
    [0.107558, 0.037150, 0.072523, 0.039312, 0.440773, 0.888883, 0.558936, 0.997920],   # W5
])
queries_y = np.array([
    9.502189400000000,   # W1
    9.632816814775900,   # W2
    9.683598182481100,   # W3
    9.617234575254900,   # W4
    9.475731837555500,   # W5
])

# Week 6 submitted — awaiting result (excluded from training)
# x_w6 = np.array([0.071236, 0.002483, 0.072505, 0.072448, 0.452562, 0.733258, 0.477203, 0.838951])

X_train = np.vstack([X_init, queries_X])
y_train = np.append(y_init, queries_y)

print(f'Total observations: {len(y_train)}')
print(f'Best observed output: {y_train.max():.6f}')
print(f'Best observed input:  {X_train[np.argmax(y_train)]}')

## 4. Data Exploration

In [ ]:
print('Output statistics:')
print(f'  Min:  {y_train.min():.6f}')
print(f'  Max:  {y_train.max():.6f}')
print(f'  Mean: {y_train.mean():.6f}')
print(f'  Std:  {y_train.std():.6f}')

# Running best across all observations
plot_running_best(y_train, title='Function 8 — Running Best Output')

### Dimension–Output Correlation

For high-dimensional functions, identifying influential dimensions allows pseudo-dimensionality reduction (fixing low-influence inputs).

In [ ]:
correlations = np.array([
    np.corrcoef(X_train[:, i], y_train)[0, 1]
    for i in range(X_train.shape[1])
])

plt.figure(figsize=(8, 4))
plt.bar([f'X{i+1}' for i in range(8)], correlations, color='steelblue')
plt.axhline(0, color='k', linewidth=0.8)
plt.ylabel('Pearson r with output')
plt.title('Function 8 — Dimension-Output Correlation')
plt.grid(axis='y'); plt.tight_layout(); plt.show()

print('Correlations (X1, X2, X3, X4, X5, X6, X7, X8):')
for i, r in enumerate(correlations):
    print(f'  X{i+1}: {r:.4f}')

## 5. Bayesian Optimisation (UCB, β=0.02)

Strategy: exploitation-heavy (trust region ±0.04)

In [ ]:
next_query, fitted_model = suggest_next_query(
    X_train=X_train,
    y_train=y_train,
    n_candidates=9_500_000,
    beta=0.02,
    length_scale=0.1,
    noise_level=1e-6,
    fit_noise=True,
    trust_region_radius=0.04,
    seed=42,
)

# Format to 6 decimal places (BBO portal requirement)
coords = ', '.join(f'{v:.6f}' for v in next_query)
print(f'Week 7 suggested query for Function 8:')
print(f'  ({coords})')

### Dimension Sensitivity (Finite-Difference Gradients)

Identifies which input dimensions most strongly influence the GP predicted mean at the current best point.

In [ ]:
best_x = X_train[np.argmax(y_train)]
sensitivity = dimension_sensitivity(fitted_model, best_x, delta=0.01)

plt.figure(figsize=(8, 4))
plt.bar([f'X{i+1}' for i in range(8)], sensitivity, color='tomato')
plt.axhline(0, color='k', linewidth=0.8)
plt.ylabel('∂μ/∂xᵢ  (GP mean gradient)')
plt.title('Function 8 — Dimension Sensitivity at Best Point')
plt.grid(axis='y'); plt.tight_layout(); plt.show()

print('Sensitivity at best point:')
for i, g in enumerate(sensitivity):
    print(f'  X{i+1}: {g:+.4f}')

## 6. Week 7 Strategy Summary

| Field | Value |
|-------|-------|
| Function | F8 (8D) |
| Total observations | n/a (computed above) |
| Acquisition | UCB, β=0.02 |
| Trust region | ±0.04 per dim |
| Strategy | Best at W3 (9.684). W5 dipped. Exploit near [0.107, 0.011, 0.211, 0.092, 0.367, 0.737, 0.442, 0.861]. |

> Submit the coordinates printed above via the BBO portal. All inputs must be in `[0, 1)` to 6 decimal places.